# Trending YouTube Video Statistics

Daily statistics for trending YouTube videos

## Download data

We download the data via `GET` request from the kaggle server. Since we download a .zip-file, we need to extract the data. In this NB, we work with US data.

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import requests

response = requests.get('https://www.kaggle.com/api/v1/datasets/download/datasnaek/youtube-new')
assert response.status_code == 200

Path("./data").mkdir(parents=True, exist_ok=True)

with ZipFile(BytesIO(response.content), "r") as zip_file:
    zip_file.extractall(
        path='./data',
        members=[
            "USvideos.csv",
            "US_category_id.json",
        ],
    )


## Read main data frame `USvideos.csv`

In [ ]:
import pandas as pd

data = pd.read_csv(
    filepath_or_buffer='./data/USvideos.csv',
    dtype={
        'channel_title': str,
        'category_id': str
    },
    parse_dates=[
        'trending_date',
        'publish_time'
    ],
    date_format={
            'trending_date': '%y.%d.%m',
            'publish_time': '%Y-%m-%dT%H:%M:%S.%fZ'
        }
    )

# Preview
data.head()

We can see some inconsistency in the data (e.g. UPPERCASE titles, quoted tags). We need to run some data cleaning before analyzing the data.

### Data Cleaning

In [ ]:
# Lower case titles, channel_titles, ect.
for col in [
    'title',
    'channel_title'
]:
    data[col] = data[col].apply(lambda x: x.lower())

# Unquote tags
data['tags'] = (
    data['tags']
    .str.replace(
        '"', '',
        regex=False
        )
    .str.split('|')
)

# Unify date cols
data['publish_date'] = pd.to_datetime(data['publish_time']).dt.normalize()


## Prepare meta data

In [ ]:
meta = pd.read_json(
    path_or_buf='./data/US_category_id.json',
    dtype={
        'id': str
    }
)

cats = pd.json_normalize(meta['items'])

cats['cat'] = cats['snippet.title'].apply(lambda x: x.lower())

# Preview
cats.head()

## Join data sets

In [ ]:
df = pd.merge(
    left=data,
    right=cats,
    left_on='category_id',
    right_on='id',
    how='left'
)

# Preview
df.head()

## Calculations

For more insights, we run some calculations

In [ ]:
# Add trend_delay
df['trend_delay'] = (df['trending_date'] - df['publish_date'])

# Add popularity
df['popularity'] = df['likes'] - df['dislikes']

## Data export

We only export the needed columns.

In [ ]:
df[[
    'video_id',
    'channel_title',
    'title',
    'tags',
    'views',
    'likes',
    'dislikes',
    'popularity',
    'publish_date',
    'trending_date',
    'trend_delay'
]].to_csv('./data/trending_youtube_video_statistics.csv')